# Lab 07｜经验技能化：怎样把一次排查经历提炼成可复用的方法？

这是专栏《自进化 Agent 实战》07 讲《经验技能化：怎样把一次排查经历提炼成可复用的方法？》的配套练习。练习使用构造的事故材料，目标是掌握处理真实场景的能力。

## 题面

我们刚处理完一次线上支付模块的事故，几个人花了几个小时才定位到根因。组里想把这套排查经验沉淀下来：以后这个模块再出告警，接手的同事能按同样的方法快速查起来，但又不会被上次的结论带偏。你会怎么设计这套沉淀和复用机制？又怎么保证新一次排查是在看当前现场，而不是沿用旧答案？

## 参考答案

我会先把“当时怎么查清的”和“当时查出了什么”分开。前者可以帮助下一位同事少走弯路，后者只说明那一次事故，不能直接作为下一次的判断。

具体提炼时，我会回看完整的排查过程，留下长期有效的方法性知识：这个模块负责什么、与哪些依赖相连，日志和发布记录去哪里查，指标含义和统计口径是什么，以及先查什么、看到什么结果再往下查。事故编号、请求编号、实例名、版本、当时的参数值和最终根因，都从可复用的方法中删掉，留在那次调查记录里。尤其不能把“上次是连接池耗尽”写成“遇到超时就先认定连接池有问题”；可以保留的是怎样用当前指标判断连接池是否异常。

落地时，我会把这些方法整理成一个模块专属的 Skill，让同事和智能体都能复用。以 Codex 为例，可以在刚完成调查的 Thread 中调用 `$skill-creator`，把方法写成 Skill。它的实现原理是：内部用 `init_skill.py` 建好目录和文件骨架，把方法写进 `SKILL.md`，再用 `quick_validate.py` 做结构校验。这两个脚本属于 Skill Creator 的内部实现。之后的新调查显式调用生成的 Skill，按里面的步骤重新取数。不过，结构校验通过只说明文件结构符合要求；查询入口是否真实可用、调查步骤是否合理、旧结论有没有变成默认判断，都不在结构校验的范围内，仍然需要人来核对。

这部分我会请熟悉支付模块的人逐条核对，自动校验负责守住结构。人要检查入口和指标口径是否仍然适用，步骤之间有没有判断依据，遇到不同结果能不能转向其他原因。这样留下来的才是一套能指导调查的方法，而不是一份换了格式的事故报告。

最后，我会选另一场原因不同的事故，在新 Thread 中试用。让智能体从当前日志、发布记录、指标和配置重新取数，再对照它的查询过程和报告，确认每个判断能对应到这次取得的材料。如果它没有查询就沿用旧根因，或者当前材料已经不支持原方向却仍照着查，就要继续修改方法。旧调查记录和每次修订的差异也要保留，方便团队追溯这套方法是怎样形成、为什么修改的。


## 准备

使用 Python 3.12 内核；需要 `GLM_API_KEY`，未设置时会弹出隐藏输入框，依赖自动补装。`lab.codex` 是真实 SDK 客户端。下面在 `Thread.turn()` 的输入中直接构造 `SkillInput(name=..., path=...)` 和 `TextInput(...)`，再调用 `run()`；这就是显式选择 Skill 并执行任务的位置。`lab.watch()` 只保护等待，`lab.record_result()` 只保存结果。


In [ ]:
import sys
from pathlib import Path
examples = next(p for base in (Path.cwd(), *Path.cwd().parents)
                for p in (base, base / "examples") if (p / "notebook07_setup.py").is_file())
sys.path.insert(0, str(examples))
from notebook07_setup import prepare
story = prepare()


## 第一幕｜Skill Creator 怎样把历史经验整理成 Skill？

把练习中已完成的调查 Thread 提炼成 Skill，观察哪些知识留下、哪些留在现场。
跑完对比“进入 Skill 的方法”与“留在原 Thread 的现场”两列，看四类知识与本次参数、根因怎样分开。
对应专栏第一幕“Skill Creator 怎样把历史经验整理成 Skill？”。


In [ ]:
import subprocess, shutil, hashlib, json
from openai_codex import ApprovalMode, Codex, Sandbox, SkillInput, TextInput
lab, work, creator = story.lab, story.work, story.creator
project, skill_dir = story.project, story.skill_dir
selected_resources = []  # 需要额外资源时，显式填入 scripts、references 或 assets
resource_args = ["--resources", ",".join(selected_resources)] if selected_resources else []
init = subprocess.run(
    [sys.executable, str(creator / "scripts/init_skill.py"), "legacy-payment-oncall",
     "--path", "legacy-payment/.agents/skills", *resource_args,
     "--interface", "display_name=历史支付模块值班排查",
     "--interface", "short_description=调查历史支付模块告警，重新核对发布、日志、连接池指标和运行配置",
     "--interface", "default_prompt=使用 $legacy-payment-oncall 调查当前告警。"], cwd=work, check=True, capture_output=True, text=True)
story.record_initialization(init)
print(init.stdout.strip())
print("第一版实际创建的目录：", [p.name for p in skill_dir.iterdir() if p.is_dir()])
story._save("initialized-directories.json", [p.name for p in skill_dir.iterdir() if p.is_dir()])
story._save("initialization-selection.json", {"resources": selected_resources})
codex: Codex = lab.codex if lab else None
session = codex.thread_start(cwd=str(work), sandbox=Sandbox.full_access,
    approval_mode=ApprovalMode.deny_all) if codex else None
if lab:
    turn = session.turn(TextInput("读取 history/old-thread.md，回顾刚完成的调查，区分现场信息与长期知识，不实施变更。"))
    with lab.watch(turn):
        history = turn.run()
    lab.record_result(turn, history, show_result=False)


In [ ]:
if lab:
    turn = session.turn([
        SkillInput(name="skill-creator", path=str(creator / "SKILL.md")),
        TextInput("使用当前 Thread 作为完整案例，创建 legacy-payment-oncall；保留模块边界、查询入口、指标含义和已验证的调查顺序。"
                  "删除事故编号、请求编号、实例参数、版本和最终根因；新调查重新读取当前日志、发布、指标和配置。"
                  "目录已初始化。第一版只保留操作说明，完善中文 SKILL.md 与 agents/openai.yaml；"
                  "资源目录只使用初始化时显式选择的项目：" + (", ".join(selected_resources) or "无")
                  + "；不添加未选择的资源目录，不修改事故材料。写入：" + str(skill_dir / "SKILL.md")),
    ])
    with lab.watch(turn):
        result = turn.run()
    lab.record_result(turn, result, show_result=False)
else:
    raise RuntimeError("离线检查不生成 Skill；请运行 run_validation.py --mode offline。")
validation = subprocess.run([sys.executable, str(creator / "scripts/quick_validate.py"), str(skill_dir)], check=True, capture_output=True, text=True)
print(validation.stdout.strip())
generated_dir = story.runtime / "generated-skill"
shutil.copytree(skill_dir, generated_dir)
story.session = session
story._save("model-draft-validation.json", story._receipt(validation))
print((generated_dir / "SKILL.md").read_text(encoding="utf-8"))
print("本轮生成的全部文件：", [str(p.relative_to(generated_dir)) for p in generated_dir.rglob("*") if p.is_file()])
story._phase("第一幕")


## 第二幕｜为什么结构合格的 Skill 仍然带着旧事故的答案？

审阅刚才实际生成的 Skill 及附件：先让同一 Thread 列出具体问题，再由你核对原文，必要时编辑本轮文件。
下一格保存生成原稿与审阅后文件的实际差异，不写入预置答案；没有修改就如实显示无差异。模型的审阅意见不代替人工判断。
对应专栏第二幕“为什么结构合格的 Skill 仍然带着旧事故的答案？”。


In [ ]:
skill_path = skill_dir / "SKILL.md"
print(skill_path.read_text(encoding="utf-8"))
turn = session.turn(TextInput("只读审阅本轮刚生成的 Skill 与它的附件，不改文件。"
    "逐条检查模块边界、查询入口、指标口径、调查顺序；指出旧根因或一次性参数是否被误写成长期规则。"
    "引用具体文件与原句，说明依据和改法；如果没有发现问题，如实说明审阅范围，不虚构缺陷。"))
with lab.watch(turn):
    review_result = turn.run()
lab.record_result(turn, review_result, show_result=False)
print(lab.clean(review_result.final_response))
(story.runtime / "review-notes.md").write_text(review_result.final_response, encoding="utf-8")
print("请核对本轮原文与意见；需要修改时，编辑上面显示的本轮 Skill 文件，再运行下一格。")

In [ ]:
import difflib
def skill_snapshot(folder):
    return {str(p.relative_to(folder)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(folder.rglob("*")) if p.is_file() and "__pycache__" not in p.parts}
original = (generated_dir / "SKILL.md").read_text(encoding="utf-8")
reviewed = skill_path.read_text(encoding="utf-8")
print("".join(difflib.unified_diff(original.splitlines(True), reviewed.splitlines(True),
    fromfile="本轮生成稿", tofile="审阅后文件")) or "SKILL.md 未修改；请确认这是审阅后的决定。")
validation = subprocess.run([sys.executable, str(creator / "scripts/quick_validate.py"), str(skill_dir)], check=True, capture_output=True, text=True)
print(validation.stdout.strip())
shutil.copytree(skill_dir, story.runtime / "reviewed-skill")
reviewed_hashes = skill_snapshot(skill_dir)
story._save("review-diff.json", {"generated": skill_snapshot(generated_dir), "reviewed": reviewed_hashes,
    "before": original, "after": reviewed, "semantic_review_required": True})
story._save("reviewed-validation.json", story._receipt(validation))
story._phase("第二幕")

## 第三幕｜新的调查还需要打开旧 Thread 吗？

练习材料构造了第二场事故：历史支付模块大量超时；在新 Thread 中调用 `legacy-payment-oncall`，从当前入口调查。
跑完对照“当前入口”“本次材料与报告”和“怎样改变判断”三列，看连接池指标、结算服务域名与运行配置怎样改变调查方向。
对应专栏第三幕“新的调查还需要打开旧 Thread 吗？”。


In [ ]:
if lab:
    if skill_snapshot(skill_dir) != reviewed_hashes:
        raise RuntimeError("Skill 在审阅记录后发生变化，请先重新审阅，而不是试用另一份文件。")
    fresh = codex.thread_start(cwd=str(project), sandbox=Sandbox.full_access,
        approval_mode=ApprovalMode.deny_all)
    turn = fresh.turn([
        SkillInput(name="legacy-payment-oncall", path=str(skill_path)),
        TextInput("使用 $legacy-payment-oncall 调查当前告警。读取 alert.md 确认范围。"
                  "用当前 Python 执行 current/query.py，后接入口名称；日志查询支持 --limit 20 与 --request-id。"
                  "按本轮 Skill 的方法重新调查，以实际取得的当前材料为依据；不要沿用历史事故的根因。"
                  "只调查与报告，不修改配置，不读取模块目录以外的旧事故记录。"
                  "用中文简短报告已确认事实、当前推断、未决问题和下一步验证方式。"),
    ])
    with lab.watch(turn):
        result = turn.run()
    lab.record_result(turn, result, show_result=False)
else:
    result = None


In [ ]:
if result:
    print(lab.clean(result.final_response))
try:
    story._save("fresh-thread.json", {"thread_id": fresh.id, "previous_thread_id": session.id,
        "skill_before": reviewed_hashes, "skill_after": skill_snapshot(skill_dir)})
    requests = lab.requests[story._phase_requests:]
    entries = ("legacy_payment_logs", "legacy_release_history", "legacy_payment_db_pool", "legacy_payment_runtime_config")
    reads = {entry: story._queried(requests, entry) for entry in entries}
    story._save("new-alert-reads.json", {"mode": "live", "queried": reads})
    (story.runtime / "new-alert-report.md").write_text(result.final_response, encoding="utf-8")
    story._table(["当前入口", "实际工具轨迹"],
        [(entry, "有成功回执" if ok else "未确认查询") for entry, ok in reads.items()],
        "结合上方本轮模型报告核对判断；未查询的入口不补成成功，报告是否成立仍需审阅。")
    story._phase("第三幕")
finally:
    story.close()